# Functions to parse and clean up ac-power data for enough-data parquet systems

## Standard early loading

(Adjust directories as needed.  This section only grabs data inside the GitHub)

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from gen_variable_standard_static import metrics_search_for_two_fragments_df
from tqdm import tqdm

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records'],
      dtype='str')

In [3]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [4]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

## Quick direct access-practice to data (can omit this section)

In [5]:
system_id = 1200
test_folder_str = f'../../../../data_ds_project/testing_yearly_parquet/{system_id}'
test_folder = Path(test_folder_str)
test_pq = pq.ParquetDataset(test_folder)
test_df = test_pq.read().to_pandas()

In [6]:
test_df.head()

,time,ac_power_kW_inverter,ac_power_kW_meter,year
0,2010-10-03 12:15:12,7.052258,NaN,2010
1,2010-10-03 12:30:12,9.621044,NaN,2010
2,2010-10-03 12:45:12,8.489045,NaN,2010
3,2010-10-03 13:00:13,9.352703,NaN,2010
4,2010-10-03 13:15:13,16.757570,NaN,2010


## Track 1: grabbing delta-t's from uncleaned data.

### Part 1a: Grabbing delta-t's in some format

In [7]:
def daily_delta_t_tracker(
    input_dir: str,
    system_id: int
):
    '''Find all daily changes for all sub-systems.
    
    Parameters:
    ------------
    input_dir: str
        The path to the results of ac_power_parquet_distiller_yearly.py
        Default is data_ds_project/testing_yearly_parquet, 
        but this may change in the final build.
    system_id: int
        The desired system_id.  
    
    Returns:
    ---------
    output_df: pd.DataFrame or None
        If no (good) data, return None
        Else, return a DataFrame with 1-2 columns (from 'inverter', 'meter', 'unknown')
        and indices corresponding to each day with significant data
        (with int data for each of year/month/day) 
    '''
    # initialize data frame
    # know all data is in 1994 - 2023 range
    num_years = 2023-1994 + 1
    num_days = num_years * 12 * 31
    output_df = pd.DataFrame(
        data = np.full((num_days, 3), fill_value=pd.NA),
        index = pd.MultiIndex.from_product(
            [range(1994, 2024), range(1, 13), range(1, 32)]
        ),
        columns = ['inverter', 'meter', 'unknown'],
        dtype='object'
    )
    output_df.index.names = ['year', 'month', 'day']
    # set target
    if input_dir[-1] == '/':
        test_folder_str = f'{input_dir}{system_id}/'
    else:
        test_folder_str = f'{input_dir}/{system_id}/'
    test_folder = Path(test_folder_str)
    # columns invariant among years accessed, so can just grab a year with no data to grab column names
    empty_year_pq = pq.ParquetDataset(
        test_folder,
        filters=[('year', '==', 1990)]
    )
    empty_year_df = empty_year_pq.read().to_pandas()

    # grab column names
    col_names = empty_year_df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    for year in range(1994, 2024):
        current_year_pq = pq.ParquetDataset(
            test_folder,
            filters=[
                ('year', '==', year)
            ]            
        )
        current_year_df = current_year_pq.read().to_pandas()
        for pow_col_name in pow_col_names:
            pow_selection = current_year_df[['time', pow_col_name]]
            # drop if no data in the column
            pow_selection = pow_selection.dropna(axis=1, how='all')
            if (pow_selection.shape[1] > 1) and (pow_selection.shape[0] > 10):
                pow_selection.loc[:, 'month'] = pow_selection['time'].dt.month
                pow_selection.loc[:, 'day'] = pow_selection['time'].dt.day
                for month in sorted(pow_selection['month'].unique()):
                    month_selection = pow_selection[pow_selection['month'] == month]
                    clean_month = int(month)
                    if (month_selection.shape[0] > 10):
                        for day in sorted(month_selection['day'].unique()):
                            day_selection = month_selection[month_selection['day'] == day]
                            clean_day = int(day)
                            if day_selection.shape[0] > 2:  # need any nonzero data, I'm afraid
                                day_selection.loc[:, 'delta_t'] = day_selection['time'].diff()
                                day_selection_common_delta = day_selection['delta_t'].value_counts().index[0]
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = day_selection_common_delta
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = day_selection_common_delta
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = day_selection_common_delta
                            elif day_selection.shape[0] == 2:  # error message
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = timedelta(hours=11, minutes=59)
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = timedelta(hours=11, minutes=59)
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = timedelta(hours=11, minutes=59)
                            elif day_selection.shape[0] == 1:
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = timedelta(hours=23, minutes=59)
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = timedelta(hours=23, minutes=59)
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = timedelta(hours=23, minutes=59)
    # drop irrelevant columns
    output_df = output_df.dropna(axis=1, how='all')
    # drop irrelevant rows
    output_df = output_df.dropna(axis=0, how='all')
    # don't bother with empty dataframes
    if output_df.shape[0] == 0:
        return None
    else:
        return output_df

Sample

In [8]:
daily_changes_1200 = daily_delta_t_tracker(
    input_dir = '../../../../data_ds_project/testing_yearly_parquet/',
    system_id=1200,
)
daily_changes_1200.head()

inverter meter
year month day                       
2010 10    3    0 days 00:15:00  <NA>
           4    0 days 00:15:00  <NA>
           5    0 days 00:15:00  <NA>
           6    0 days 00:15:00  <NA>
           7    0 days 00:15:00  <NA>

Need a variant to extract directly from a given dataframe with a "date" column.

In [9]:
def daily_delta_t_from_df(
    simple_df: pd.DataFrame
):
    '''Grab the daily mode of delta-t's.
    
    Parameters:
    -----------
    simple_df: pandas.DataFrame
       A DataFrame with "time" and "date" columns.

    Returns:
    ---------
    output_ser: pandas.Series
       Series with name 'daily_common_timestep'
       indicating the dailty commonest delta_t.
       Omits days with few data points.
    '''
    time_only_df = simple_df[['time', 'date']].copy(deep=True)
    dates_collection = time_only_df['date'].unique()
    output_ser = pd.Series(
        data = np.full(len(dates_collection), pd.NA),
        index = dates_collection,
        name = 'daily_common_timestep'
    )
    for given_date in dates_collection:
        date_selection = time_only_df[time_only_df['date'] == given_date]
        num_data_points = date_selection.shape[0]
        if num_data_points >= 10:  # now only want days with good data
            date_selection.loc[:, 'delta_t'] = date_selection['time'].diff()
            date_common_diff = date_selection['delta_t'].value_counts().index[0]
            output_ser.at[given_date] = date_common_diff
    output_ser = output_ser.dropna()  # drop definitely-bad points
    return output_ser

### Part 1b: Extracting the dates with significant changes in common delta_t

In [10]:
def extract_delta_t_changes(
    input_dir: str,
    system_id: int,
    na_drop_type: str = 'leading_trailing',
):
    '''Extract the list of days with significant data that flank a change in delta_t.
    (Note: this can be a sudden change or an after-5-months-of-no-data change.)
    Parameters:
    ------------
    input_dir: str
        The path to the results of ac_power_parquet_distiller_yearly.py
        Default is data_ds_project/testing_yearly_parquet/, 
        but this may change in the final build.
    system_id: int
        The desired system_id.
    na_drop_type: str,
        If na_drop_type == "all", will drop all na terms
        If na_drop_type == "leading_trailing", will retain interior na's
        and drop those outside that.
    
    Returns:
    ---------
    out_list: list[pandas.Series]
       A list with one element if one power-attribute,
       and two elements if two power-attributes.
       Each term is a pandas Series object with the dates and common-delta-t's
       flanking a change in the delta-t.
       Again, the dates are in an awkward year/month/day index format
       and not.
    '''
    out_list = []
    my_daily_terms = daily_delta_t_tracker(input_dir=input_dir,
        system_id=system_id
    )
    for col in my_daily_terms.columns:
        col_forwards = f'{col}_forward_difference'
        col_backwards = f'{col}_backward_difference'
        my_daily_terms.loc[:, col_forwards] = -my_daily_terms[col].diff(periods=-1)
        my_daily_terms.loc[:, col_backwards] = my_daily_terms[col].diff(periods=1)
        # grab relevant columns and drop_na
        my_col_relevant = my_daily_terms[[col, col_forwards, col_backwards]].copy(deep=True)
        # first and last rows will always have NaT terms, so grab NaT rows
        # and any terms with large differences
        my_col_relevant = my_daily_terms[
            my_daily_terms[col_forwards].isna()
            | my_daily_terms[col_backwards].isna()
            | ((my_daily_terms[col_forwards] - my_daily_terms[col_backwards]) > timedelta(seconds=15))
            | ((my_daily_terms[col_backwards] - my_daily_terms[col_forwards]) > timedelta(seconds=15))
        ]
        return_column = my_col_relevant[col]
        # now drop NA's as required
        if na_drop_type == 'all':
            return_column = return_column.dropna()
        elif na_drop_type == 'leading_trailing':
            first_ind = return_column.first_valid_index()
            last_ind = return_column.last_valid_index()
            return_column = return_column.loc[first_ind:last_ind]
        out_list.append(return_column)
    return out_list

#### Variant code where you just input the 1a results

In [11]:
def extract_delta_t_changes_mod(
    daily_changes_df: pd.DataFrame,
    na_drop_type: str = 'leading_trailing',
):
    '''Extract the list of days with significant data that flank a change in delta_t,
    without redoing previous part.
    (Note: this can be a sudden change or an after-5-months-of-no-data change.)

    Parameters:
    ------------
    daily_changes_df: pandas.DataFrame
        The results of daily_delta_t_tracker(*args)
    na_drop_type: str,
        If na_drop_type == "all", will drop all na terms
        If na_drop_type == "leading_trailing", will retain interior na's
        and drop those outside that.
    
    Returns:
    ---------
    out_list: list[pandas.Series]
       A list with one element if one power-attribute,
       and two elements if two power-attributes.
       Each term is a pandas Series object with the dates and common-delta-t's
       flanking a change in the delta-t.
       Again, the dates are in an awkward year/month/day index format
       and not.
    '''
    out_list = []
    my_daily_terms = daily_changes_df.copy(deep=True)
    for col in my_daily_terms.columns:
        col_forwards = f'{col}_forward_difference'
        col_backwards = f'{col}_backward_difference'
        my_daily_terms.loc[:, col_forwards] = -my_daily_terms[col].diff(periods=-1)
        my_daily_terms.loc[:, col_backwards] = my_daily_terms[col].diff(periods=1)
        # grab relevant columns and drop_na
        my_col_relevant = my_daily_terms[[col, col_forwards, col_backwards]].copy(deep=True)
        # first and last rows will always have NaT terms, so grab NaT rows
        # and any terms with large differences
        my_col_relevant = my_daily_terms[
            my_daily_terms[col_forwards].isna()
            | my_daily_terms[col_backwards].isna()
            | ((my_daily_terms[col_forwards] - my_daily_terms[col_backwards]) > timedelta(seconds=15))
            | ((my_daily_terms[col_backwards] - my_daily_terms[col_forwards]) > timedelta(seconds=15))
        ]
        return_column = my_col_relevant[col]
        # now drop NA's as required
        if na_drop_type == 'all':
            return_column = return_column.dropna()
        elif na_drop_type == 'leading_trailing':
            first_ind = return_column.first_valid_index()
            last_ind = return_column.last_valid_index()
            return_column = return_column.loc[first_ind:last_ind]
        out_list.append(return_column)
    return out_list

Sample

In [12]:
delta_changes_1200 = extract_delta_t_changes(
    input_dir='../../../../data_ds_project/testing_yearly_parquet/',
    system_id=1200,
    na_drop_type='all')
for df in delta_changes_1200:
    print(df)

year  month  day
2010  10     3      0 days 00:15:00
2011  7      20     0 days 00:15:00
             21     0 days 00:05:00
             25     0 days 00:05:00
             26     0 days 00:15:00
      12     22     0 days 00:15:00
             23     0 days 00:05:00
2013  11     28     0 days 00:05:00
             29     0 days 00:03:00
      12     31     0 days 00:03:00
Name: inverter, dtype: object
year  month  day
2013  1      20     0 days 00:05:00
      11     28     0 days 00:05:00
             29     0 days 00:03:00
2016  1      25     0 days 00:03:00
             26     0 days 00:05:00
2020  7      26     0 days 00:05:00
Name: meter, dtype: object


### Part 1(c): Whoops, I really need to index by datetime.date objects for interoperability with a future part!

In [13]:
def daily_delta_t_reshaper(
    input_dir: str,
    system_id: int
):
    '''Rewrite previous steps to have a date column,
    and extract a list of the change-dates as well.
    
    Parameters:
    ------------
    input_dir: str
        The path to the results of ac_power_parquet_distiller_yearly.py
        Default is data_ds_project/testing_yearly_parquet/, 
        but this may change in the final build.
    system_id: int
        The desired system_id.
    
    Returns:
    -------------
    unnamed: list[tuple[pandas.DataFrame, pandas.Series, list]]
       A list with one or two terms, depending on how many power-terms we track
       Each list is of the form
       (timesteps, changes, bad_days),
       where timesteps is the daily_delta_t_tracker(input_dir, system_id) result
       with old index replaced by datetime.date objects,
       changes is the extract_delta_t_changes(*args) result for the self-same group,
       and bad_days is the list of dates in changes (as datetime.date objects).  
    '''
    # start by grabbing all the daily deltas with or without good data
    time_steps_daily = daily_delta_t_tracker(
        input_dir=input_dir,
        system_id=system_id
    )
    system_data_types = time_steps_daily.columns 
    if len(system_data_types) == 1:
        first_timesteps = time_steps_daily[[system_data_types[0]]]
        second_timesteps = None
    elif len(system_data_types) == 2:
        first_timesteps = time_steps_daily[[system_data_types[0]]]
        second_timesteps = time_steps_daily[[system_data_types[1]]]
    else:
        print(system_data_types)
        raise ValueError(f'Bad results from daily_changes_no_oracle({system_id})')
    # reset time
    first_timesteps = first_timesteps.reset_index()
    first_timesteps.loc[:, 'date'] = first_timesteps.apply(
        lambda row: date(row['year'], row['month'], row['day']),
        axis = 1
    )
    first_timesteps = first_timesteps.set_index('date')
    first_timesteps = first_timesteps.drop(columns = ['year', 'month', 'day'])
    if (second_timesteps is not None) and (second_timesteps.shape[0] > 0):
        second_timesteps = second_timesteps.reset_index()
        second_timesteps.loc[:, 'date'] = second_timesteps.apply(
            lambda row: date(row['year'], row['month'], row['day']),
            axis = 1
        )
        second_timesteps = second_timesteps.set_index('date')
        second_timesteps = second_timesteps.drop(columns = ['year', 'month', 'day'])
    # grab bad deltas
    change_days = extract_delta_t_changes(input_dir,
        system_id, na_drop_type='leading_trailing'
    )
    if len(change_days) == 1:
        first_changes = change_days[0]
        second_changes = None
    elif len(change_days) == 2:
        first_changes = change_days[0]
        second_changes = change_days[1]
    # reset time
    first_changes = first_changes.reset_index()
    first_changes.loc[:, 'date'] = first_changes.apply(
        lambda row: date(row['year'], row['month'], row['day']),
        axis = 1
    )
    first_changes = first_changes.drop(
        columns=['year', 'month', 'day']
    )
    first_bad_dates = sorted(first_changes['date'].unique())
    first_changes = first_changes.set_index('date')
    if (second_changes is not None) and (second_changes.shape[0] > 0):
        second_changes = second_changes.reset_index()
        second_changes.loc[:, 'date'] = second_changes.apply(
            lambda row: date(row['year'], row['month'], row['day']),
            axis = 1
        )
        second_changes = second_changes.drop(
            columns=['year', 'month', 'day']
        )
        second_bad_dates = sorted(second_changes['date'].unique())
        second_changes = second_changes.set_index('date')
    if (second_timesteps is not None) and (second_timesteps.shape[0] > 0):
        return [(first_timesteps, first_changes, first_bad_dates),
                (second_timesteps, second_changes, second_bad_dates)]
    else:
        return [(first_timesteps, first_changes, first_bad_dates),]

Sample

In [14]:
changes_group = daily_delta_t_reshaper(
    input_dir='../../../../data_ds_project/testing_yearly_parquet/',
    system_id=1200
)
inv_delta_ts, inv_changes, inv_bad_dates = changes_group[0]
met_delta_ts, met_changes, met_bad_dates = changes_group[1]
print(inv_delta_ts.head())
print(inv_changes)
print(inv_bad_dates)
print(met_delta_ts.head())
print(met_changes)
print(met_bad_dates)

                   inverter
date                       
2010-10-03  0 days 00:15:00
2010-10-04  0 days 00:15:00
2010-10-05  0 days 00:15:00
2010-10-06  0 days 00:15:00
2010-10-07  0 days 00:15:00
                   inverter
date                       
2010-10-03  0 days 00:15:00
2011-07-20  0 days 00:15:00
2011-07-21  0 days 00:05:00
2011-07-25  0 days 00:05:00
2011-07-26  0 days 00:15:00
2011-12-22  0 days 00:15:00
2011-12-23  0 days 00:05:00
2013-11-28  0 days 00:05:00
2013-11-29  0 days 00:03:00
2013-12-31  0 days 00:03:00
[datetime.date(2010, 10, 3), datetime.date(2011, 7, 20), datetime.date(2011, 7, 21), datetime.date(2011, 7, 25), datetime.date(2011, 7, 26), datetime.date(2011, 12, 22), datetime.date(2011, 12, 23), datetime.date(2013, 11, 28), datetime.date(2013, 11, 29), datetime.date(2013, 12, 31)]
           meter
date            
2010-10-03  <NA>
2010-10-04  <NA>
2010-10-05  <NA>
2010-10-06  <NA>
2010-10-07  <NA>
                      meter
date                       
2013-01

In [102]:
changes_group_50 = daily_delta_t_reshaper(
    input_dir='../../../../data_ds_project/testing_yearly_parquet/',
    system_id=50
)
len(changes_group_50)

1

In [103]:
timesteps_50, changes_50, bad_dates_50 = changes_group_50[0]

## Part 2: Extract data per year (or a little extra) and clean out zeros.

In [17]:
def standardize(df: pd.DataFrame,
                null_or_zero: str,
                system_id: int,
                systems_cleaned: pd.DataFrame,
                drop_na: bool):
    '''Standardize input dataframes for univariate analysis.
    
    Parameters
    -----------
    df: pandas.DataFrame
        The dataframe output from ac_power_parquet_distiller_yearly.py
        or ac_power_parquet_distiller.py
    null_or_zero: str, "null", "zero", or "raw"
        Determine what to do with data less than 1 percent
        of maximum value
        If "null", replace small values by numpy.nan
        If "zero", replace with zero.
        If "raw", skip this cleaning step
        If anything else, throw a ValueError.
    system_id: int
        The system_id of the system.
    systems_cleaned: pd.DataFrame
        The metadata dataframe.
    drop_na: bool
        Determine whether or not to drop NA terms

    Returns
    ---------
    A list of one or two DataFrames.
    If both inverter and meter in input, get a list of two DataFrames
    If only one power input, get a list of one DataFrame
    Otherwise, bad input, ValueError
    '''
    # copy input for future use.
    df = df.copy(deep=True)
    # grab column names
    col_names = df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    # grab official max. value.
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_index = relevant_rows_systems.index[0]
    max_dc_capacity = relevant_rows_systems.loc[first_index, 'dc_capacity_kW']
    for pow_col_name in pow_col_names:
        local_max = df[pow_col_name].max()
        smaller_max = max(min(local_max, max_dc_capacity), 0)
        lower_bound = 0.01 * smaller_max
        if null_or_zero == 'null':
            df[pow_col_name] = df.apply(
                lambda row: row[pow_col_name]
                if (row[pow_col_name] is not pd.NA
                    and row[pow_col_name] is not np.nan
                    and row[pow_col_name] >= lower_bound)
                else np.nan,  # not pandas.NA with float64 dtype
                axis = 1
            )
        elif null_or_zero == 'zero':
            df[pow_col_name] = df.apply(
                lambda row: row[pow_col_name]
                if (row[pow_col_name] is not pd.NA
                    and row[pow_col_name] is not np.nan
                    and row[pow_col_name] >= lower_bound)
                else 0,
                axis = 1
            )
        elif null_or_zero == 'raw':
            pass
        else:
            raise ValueError(f'null_or_zero, input {null_or_zero}, is none of'
                             + '"null", "zero", or "raw".')
    # if both inverter and meter, make two columns
    if (
        any(pow_col_names.str.contains('inv'))
        and any(pow_col_names.str.contains('met'))
    ):
        # need two frames
        inv_pow_names = pow_col_names[pow_col_names.str.contains('inv')]
        if len(inv_pow_names) == 1:
            inv_pow_name = inv_pow_names[0]
        else:
            raise ValueError('Bad input dataframe -- multiple inverter columns!')
        met_pow_names = pow_col_names[pow_col_names.str.contains('met')]
        if len(met_pow_names) == 1:
            met_pow_name = met_pow_names[0]
        else:
            raise ValueError('Bad input dataframe -- multiple meter columns!')
        inv_data = df[['time', inv_pow_name]].copy(deep=True)
        met_data = df[['time', met_pow_name]].copy(deep=True)
        inv_data = inv_data.drop_duplicates()
        met_data = met_data.drop_duplicates()
        if drop_na:
            inv_data = inv_data.dropna()
            met_data = met_data.dropna()
        # rename columns
        renamer = {
            inv_pow_name: 'power',
            met_pow_name: 'power'
        }
        inv_data = inv_data.rename(columns=renamer)
        met_data = met_data.rename(columns=renamer)
        return [inv_data, met_data]
    elif len(pow_col_names) == 1:
        pow_col_name = pow_col_names[0]
        my_data = df[['time', pow_col_name]].copy(deep=True)
        renamer = {
            pow_col_name: 'power',
        }
        my_data = my_data.rename(columns=renamer)
        my_data = my_data.drop_duplicates()
        if drop_na:
            my_data = my_data.dropna()
        return [my_data,]
    else:
        raise ValueError('Not expected input!')


def year_data_extractor(
    input_dir: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    year: int
):
    '''Grab data from the specified systems for the given year
    (perhaps a little more if required)
    and standardize it.

    Parameters:
    -------------
    input_dir: str
        The path to the results of ac_power_parquet_distiller_yearly.py
        Default is data_ds_project/testing_yearly_parquet/, 
        but this may change in the final build.
    system_id: int
        The desired system_id.
    systems_cleaned: pd.DataFrame
        The metadata dataframe.
    year: int
        The desired year
    
    Returns:
    -----------
    unnamed: tuple[pandas.DataFrame]
        A tuple with 1 or 2 elements, depending on the system
        Each DataFrame has 4 columns:
        "time": measurement time
        "power": measuremed power (in kW)
        "date": measurement date
        "count" number of large-enough data points for that day.
        
    '''

    # grab yearly data, with a month before and after (as needed)
    if input_dir[-1] == '/':
        test_folder_str = f'{input_dir}{system_id}/'
    else:
        test_folder_str = f'{input_dir}/{system_id}/'
    test_folder = Path(test_folder_str)
    current_year_pq = pq.ParquetDataset(
        test_folder,
        filters=[('year', '==', year)]
    )
    current_year_df = current_year_pq.read().to_pandas()

    if current_year_df.shape[0] < 10:
        return None
    else:
        # drop too-small data
        cleaned_data = standardize(
            current_year_df,
            'null',
            system_id,
            systems_cleaned,
            True
        )
        if len(cleaned_data) == 1:
            first_cleaned = cleaned_data[0]
            second_cleaned = None
        elif len(cleaned_data) == 2:
            first_cleaned = cleaned_data[0]
            second_cleaned = cleaned_data[1]
        else:
            raise ValueError('Output of standardize(*args) has the wrong dimensions.')
        # grab dates and counts
        first_cleaned.loc[:, 'date'] = first_cleaned['time'].dt.date
        first_cleaned.loc[:, 'count'] = first_cleaned.groupby(['date'])['power'].transform('count')
        if (second_cleaned is not None) and (second_cleaned.shape[0] > 0):
            second_cleaned.loc[:, 'date'] = second_cleaned['time'].dt.date
            second_cleaned.loc[:, 'count'] = second_cleaned.groupby(['date'])['power'].transform('count')
        # return
        if (second_cleaned is not None) and (second_cleaned.shape[0] > 0):
            return (first_cleaned, second_cleaned)
        else:
            return (first_cleaned,)

In [18]:
data_1200_year_2013 = year_data_extractor(
    input_dir='../../../../data_ds_project/testing_yearly_parquet/',
    system_id=1200,
    systems_cleaned=systems_cleaned,
    year=2013
)
data_1200_inv_2013, data_1200_met_2013 = data_1200_year_2013
print(data_1200_inv_2013.head())
print(data_1200_met_2013.tail())

                 time     power        date  count
0 2013-01-20 16:25:32  3.271555  2013-01-20      9
1 2013-01-20 16:30:33  3.039195  2013-01-20      9
2 2013-01-20 16:35:33  2.757586  2013-01-20      9
3 2013-01-20 16:40:33  2.424857  2013-01-20      9
4 2013-01-20 16:45:33  2.048286  2013-01-20      9
                     time  power        date  count
50527 2013-12-31 17:12:00   1.32  2013-12-31    181
50528 2013-12-31 17:15:00   1.04  2013-12-31    181
50529 2013-12-31 17:18:00   0.82  2013-12-31    181
50530 2013-12-31 17:21:00   0.62  2013-12-31    181
50531 2013-12-31 17:24:00   0.44  2013-12-31    181


In [107]:
def basic_days_per_year_count(
    input_dir: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    start_year: int = 1994,
    end_year: int = 2023
):
    if input_dir[-1] == '/':
        test_folder_str = f'{input_dir}{system_id}/'
    else:
        test_folder_str = f'{input_dir}/{system_id}/'
    test_folder = Path(test_folder_str)
    empty_pq = pq.ParquetDataset(
        test_folder,
        filters = [('year', '==', 1990)]
    )
    empty_df = empty_pq.read().to_pandas()
    num_cols = len(empty_df.columns) - 2  # remove 'time', 'year' columns
    num_years = end_year - start_year + 1
    day_counts_per_year = pd.DataFrame(
        data = np.zeros(shape=(num_years, num_cols)),
        index = range(start_year, end_year + 1),
        columns = ['first_reading', 'second_reading'][0:num_cols],
        dtype = 'int32'
    )
    for year in range(start_year, end_year + 1):
        yearly_data = year_data_extractor(
            input_dir=input_dir,
            system_id=system_id,
            systems_cleaned=systems_cleaned,
            year=year
        )
        if yearly_data is None:
            continue
        elif len(yearly_data) == 1:
            day_counts_per_year.at[year, 'first_reading'] = yearly_data[0]['date'].nunique()
        elif len(yearly_data) == 2:
            day_counts_per_year.at[year, 'first_reading'] = yearly_data[0]['date'].nunique()
            day_counts_per_year.at[year, 'second_reading'] = yearly_data[1]['date'].nunique()
    # drop unneeded rows
    day_counts_per_year = day_counts_per_year.dropna(axis=0, how='all')
    return day_counts_per_year


In [108]:
basic_days_per_year_count(
    '../../../../data_ds_project/testing_yearly_parquet/',
    36,
    systems_cleaned,
    2012,
    2019
)

,first_reading
2012,197
2013,317
2014,351
2015,357
2016,340
2017,356
2018,293
2019,165


In [109]:
basic_days_per_year_count(
    '../../../../data_ds_project/testing_yearly_parquet/',
    50,
    systems_cleaned,
    1994,
    2023
)

,first_reading
1994,232
1995,353
1996,337
1997,296
1998,307
1999,190
2000,161
2001,313
2002,270
2003,241


In [21]:
data_36_year_2013, = year_data_extractor(
    input_dir='../../../../data_ds_project/testing_yearly_parquet/',
    system_id=36,
    systems_cleaned=systems_cleaned,
    year=2013
)

In [ ]:
data_

## Step 3: Compare your results for parts (a) and (b) and find the good dates.

In [111]:
def grab_close_time_intervals(row):
    possible_gen_columns = ['inverter', 'meter', 'unknown']
    for col_name in possible_gen_columns:
        try:
            row[col_name]
            right_col_name = col_name
            break
        except KeyError:
            continue
        except BaseException as e:
            raise e
    if (
        (row[right_col_name] - row['daily_common_timestep']) <= timedelta(seconds = 15)
        and (row['daily_common_timestep'] - row[right_col_name]) <= timedelta(seconds=15)
    ):
        return True
    else:
        return False


def reconcile_deltas(
    cleaned_data: pd.DataFrame,
    general_diffs: pd.DataFrame,
    bad_dates: list,
    is_strict: bool
):
    '''Take the cleaned data and try to get a "good lump" of timescales per day.
    
    Parameters:
    ------------
    cleaned_data: pandas.DataFrame
        One of the terms from year_data_extractor(*args)
    general_diffs: pd.DataFrame
        The dictionary of deltas-per-day, as rewritten by
        daily_delta_t_reshaper(*args)
    bad_dates: list[datetime.date]
        The list of datetimes of potential data-change days.
        Again, as rewritten by daily_delta_t_reshaper(*args)
    is_strict: bool
        If False, keep data as long as it is the endpoint for at least one
        appropriate-difference column.
        If True, keep data if it both starts and ends delta_t's
        [except for the very start and end]

    Returns:
    ---------
    cleaned_data: pandas.DataFrame or None
        A trimmed version of the original cleaned data,
        or None if no appropriate data found.
    '''
    # make sure 'date' and 'count' columns exist.
    if 'date' not in cleaned_data.columns:
        cleaned_data.loc[:, 'date'] = cleaned_data['time'].dt.date
    if 'count' not in cleaned_data.columns:
        cleaned_data.loc[:, 'count'] = cleaned_data.groupby(['date'])['power'].transform('count')
    if cleaned_data.shape[0] == 0:
        raise ValueError('Bad cleaned_data input!')
    # make sure cleaned commmon delta_t and raw common delta_t match!
    # else, too much data to deal with.
    diffs_col_name = general_diffs.columns[-1]
    deltas_of_cleaned = daily_delta_t_from_df(cleaned_data)
    if deltas_of_cleaned is None or deltas_of_cleaned.shape[0] == 0:
        print('No data pre-merge')
        return None
    compare_timesteps = pd.merge(general_diffs, deltas_of_cleaned,
                                 how='right',
                                 left_index=True, right_index=True,
                                 suffixes=['_general', '_cleaned'])
    # check for nones
    if compare_timesteps.shape[0] == 0:
        print('No data post-merge')
        return None
    compare_timesteps.loc[:, 'matched'] = compare_timesteps.apply(
        grab_close_time_intervals,
        axis=1
    )
    compare_timesteps = compare_timesteps[compare_timesteps['matched']]
    compare_timesteps_dates = compare_timesteps.index.unique()
    cleaned_data = cleaned_data[cleaned_data['date'].isin(compare_timesteps_dates)]
    cleaned_data = cleaned_data[~cleaned_data['date'].isin(bad_dates)]
    if cleaned_data.shape[0] == 0:
        print('No data post-pruning')
        return None
    min_date = cleaned_data['date'].min()
    max_date = cleaned_data['date'].max()
    num_bad_dates = len(bad_dates)
    for j in range(num_bad_dates - 1):
        start_run = bad_dates[j]
        end_run = bad_dates[j + 1]
        # if irrelevant dates, nothing to do
        # for error-checking here, a somewhat weird try statement.
        try:
            irrelevancy_cond = (end_run <= min_date) or (start_run >= max_date)
        except TypeError as e:
            print(f'start_run is {start_run}')
            print(f'end_run is {end_run}')
            print(f'min_date is {min_date}')
            print(f'max_date is {max_date}')
            raise e
        except BaseException as e:
            raise e
        # if irrelevant range or a short run, no need to study it
        if irrelevancy_cond:
            continue
        # find a good delta for this selection
        run_selection = cleaned_data[
                (cleaned_data['date'] > start_run)
                & (cleaned_data['date'] < end_run)
            ]
        if (
            ((end_run - start_run) < timedelta(days=4))
            or (run_selection.shape[0] < (24*3))
        ):
            # too short a run, drop it!
            cleaned_data.drop(index=run_selection.index)
            continue
        else:
            temp_start_run = start_run + timedelta(days=1)
            if temp_start_run < min_date:
                temp_start_run = min_date
            have_good_delta = False
            num_attempts = 0
            while not have_good_delta:
                try: 
                    good_delta = general_diffs.at[temp_start_run, diffs_col_name]
                    have_good_delta = True
                except KeyError:
                    temp_start_run += timedelta(days=1)
                    num_attempts += 1
                    if (num_attempts > 50) or (temp_start_run == end_run):
                        # attempt to localize the error.
                        print(f'start_run is {start_run}')
                        print(f'end_run is {end_run}')
                        print(f'min_date is {min_date}')
                        print(f'max_date is {max_date}')
                        raise RuntimeError("Can't find a good delta!")
        # rotate through the days in this group
        for date in sorted(run_selection['date'].unique()):
            day_selection = run_selection[run_selection['date'] == date]
            if day_selection.shape[0] > 1:
                day_selection.loc[:, 'forward_diff'] = -day_selection['time'].diff(periods=-1)
                day_selection.loc[:, 'backward_diff'] = day_selection['time'].diff(periods=1)
                # find data with bad time-frames
                if is_strict:
                    # bad_data if either "before" or "after" is bad delta_t
                    bad_data = day_selection[
                        (
                            (day_selection['forward_diff'] > (good_delta + timedelta(seconds = 15)))
                            | (day_selection['forward_diff'] < (good_delta - timedelta(seconds = 15)))
                        )
                        | (
                            (day_selection['backward_diff'] > (good_delta + timedelta(seconds = 15)))
                            | (day_selection['backward_diff'] < (good_delta -  timedelta(seconds=15)))
                        )
                    ]
                    # debug
                    day_selection = day_selection.drop(index = bad_data.index)
                    # re-run differences.  All of one kind?
                    day_selection.loc[:, 'forward_diff'] = day_selection['time'].diff(periods=-1)
                    bad_selection = day_selection[
                        day_selection['forward_diff'] > (1.9 * good_delta)
                    ]
                    bad_day = (bad_selection.shape[0] > 0)
                else:
                    # bad data if neither forward_difference nor backward_difference is good.
                    bad_data = day_selection[
                        (
                            (day_selection['forward_diff'] > (good_delta + timedelta(seconds = 15)))
                            | (day_selection['forward_diff'] < (good_delta - timedelta(seconds = 15)))
                        )
                        & (
                            (day_selection['backward_diff'] > (good_delta + timedelta(seconds = 15)))
                            | (day_selection['backward_diff'] < (good_delta -  timedelta(seconds=15)))
                        )
                    ]
                    # old criteria, so
                    bad_day = False
                if bad_day:
                    cleaned_data.drop(index=day_selection.index)
                else:
                    cleaned_data.drop(index=bad_data.index)
            elif day_selection.shape[0] == 1:
                cleaned_data.drop(index=day_selection.index)
    return cleaned_data


In [39]:
data_1200_inv_2013_trimmed = reconcile_deltas(data_1200_inv_2013, inv_delta_ts, inv_bad_dates, True)
data_1200_inv_2013_less_trimmed = reconcile_deltas(data_1200_inv_2013, inv_delta_ts, inv_bad_dates, False)

In [25]:
data_1200_inv_2013_less_trimmed.head()

,time,power,date,count
22,2013-01-21 07:55:31,0.652409,2013-01-21,108
23,2013-01-21 08:00:31,0.850692,2013-01-21,108
24,2013-01-21 08:05:31,1.207614,2013-01-21,108
25,2013-01-21 08:10:31,1.573733,2013-01-21,108
26,2013-01-21 08:15:32,1.895800,2013-01-21,108


In [26]:
print(data_1200_inv_2013.shape)
print(data_1200_inv_2013_less_trimmed.shape)
print(data_1200_inv_2013_trimmed.shape)

(31297, 4)
(30848, 4)
(30848, 4)


In [40]:
data_36_yr_2013_trimmed = reconcile_deltas(data_36_year_2013, timesteps_36, bad_dates_36, True)

Try building up rather than breaking down.

## Step 4: Put it all together and grab all the good data.


In [110]:
def good_data_identifier(
    input_dir,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    return_type: str,
):
    '''Grab good data for all years for a given system.
    
    Parameters:
    -------------
    input_dir: str
        The path to the results of ac_power_parquet_distiller_yearly.py
        Default is data_ds_project/testing_yearly_parquet/, 
        but this may change in the final build.
    system_id: int
        The desired system_id.
    systems_cleaned: pd.DataFrame
        The metadata dataframe.
    return_type: str, "date" or "time_power"
        If return_type is "date", return a list of dates for each subsystem.
        If return_type is "time_power, return a pandas DataFrame with time/power columns
        and cleaned-data rows
    '''
    reshaped_data = daily_delta_t_reshaper(
        input_dir=input_dir,
        system_id=system_id
    )
    first_timesteps, first_changes, first_bad_dates = reshaped_data[0]
    if len(reshaped_data) == 2:
        second_timesteps, second_changes, second_bad_dates = reshaped_data[1]
    else:
        second_timesteps = None
        second_changes = None
        second_bad_dates = None
    # prepeare return containers
    if return_type == 'date':
        first_cleaned_dates = []
        second_cleaned_dates = []
    elif return_type == 'time_power':
        first_cleaned_compendium = []
        second_cleaned_compendium = []
    else:
        raise ValueError(
            f'return_type is {return_type},'
            + 'not the valid choices of "date" or "time_power".'
        )
    for year in range(1994, 2024):
        cleaned_data = year_data_extractor(
            input_dir=input_dir,
            system_id=system_id,
            systems_cleaned=systems_cleaned,
            year=year
        )
        if cleaned_data is None:
            first_cleaned = None
            second_cleaned = None
            continue
        elif len(cleaned_data) == 1:
            first_cleaned = cleaned_data[0]
            second_cleaned = None
        elif len(cleaned_data) == 2:
            first_cleaned = cleaned_data[0]
            second_cleaned = cleaned_data[1]
        else:
            raise ValueError('Trouble with just_over_year_data_extractor(*args)')
        if (first_cleaned is not None) and (first_cleaned.shape[0] > 10):
            first_cleaned = reconcile_deltas(
                first_cleaned,
                first_timesteps,
                first_bad_dates,
                True
            )
            if first_cleaned is not None:
                if return_type == 'date':
                    first_cleaned_dates += sorted(first_cleaned['date'].unique())
                elif return_type == 'time_power':
                    first_cleaned_compendium.append(first_cleaned[['time', 'power']])
        if (second_cleaned is not None) and (second_cleaned.shape[0] > 10):
            second_cleaned = reconcile_deltas(
                second_cleaned,
                second_timesteps,
                second_bad_dates,
                True
            )
            if second_cleaned is not None:
                if return_type == 'date':
                    second_cleaned_dates += sorted(second_cleaned['date'].unique())
                elif return_type == 'time_power':
                    second_cleaned_compendium.append(second_cleaned[['time', 'power']])
    if return_type == 'date':
        if len(first_cleaned_dates) > 0:
            if len(second_cleaned_dates) > 0:
                return (first_cleaned_dates, second_cleaned_dates)
            else:
                return (first_cleaned_dates,)
        else:
            raise ValueError(f'System {system_id} does not appear to have any data!')
    elif return_type == 'time_power':
        if len(first_cleaned_compendium) > 0:
            first_cleaned_total = pd.concat(first_cleaned_compendium)
        else:
            raise ValueError(f'System {system_id} does not appear to have any data!')
        if len(second_cleaned_compendium) > 0:
            second_cleaned_total = pd.concat(second_cleaned_compendium)
        else:
            second_cleaned_total = None
        if (second_cleaned_total is not None):
            return (first_cleaned_total, second_cleaned_total)
        elif (first_cleaned_total is not None):
            return (first_cleaned_total,)
        else:
            return None
    else:
        return None


In [22]:
good_data_1200 = good_data_identifier(
    input_dir='../../../../data_ds_project/testing_yearly_parquet/',
    system_id=1200,
    systems_cleaned=systems_cleaned,
    return_type='time_power'
)

In [23]:
good_data_1200_inv, good_data_1200_met = good_data_1200

In [24]:
good_data_1200_inv.head()

,time,power
29,2010-10-04 08:15:44,0.573328
30,2010-10-04 08:30:44,0.915454
31,2010-10-04 08:45:44,0.954552
32,2010-10-04 09:00:44,1.131823
33,2010-10-04 09:15:44,1.540468


In [25]:
good_data_1200_met.head()

,time,power
35212,2013-11-30 07:21:00,0.42
35213,2013-11-30 07:24:00,0.50
35215,2013-11-30 07:27:00,0.62
35216,2013-11-30 07:30:00,0.80
35218,2013-11-30 07:33:00,1.06


In [44]:
good_dates_inv_1200, good_dates_met_1200 = good_data_identifier(
    input_dir='../../../../data_ds_project/testing_yearly_parquet/',
    system_id=1200,
    systems_cleaned=systems_cleaned,
    return_type='date'
)

In [45]:
len(good_dates_inv_1200)

903

In [46]:
len(good_dates_met_1200)

2044

#### For reference, find the runs of consecutive days

In [94]:
def good_runs(my_dates):
    runs_list = []
    num_dates = len(my_dates)
    if num_dates == 0:
        return []
    elif num_dates == 1:
        return [my_dates,]
    else:
        dates_ser = pd.Series(
            data = my_dates,
            name = 'date',
        )
        dates_ser = dates_ser.sort_values()
        dates_df = pd.DataFrame(dates_ser)
        dates_df.loc[:, 'forward_diff'] = -dates_df['date'].diff(periods=-1)
        dates_df.loc[:, 'good_forward'] = dates_df.apply(
            lambda row: row['forward_diff'] == timedelta(days=1),
            axis=1
        )
        end_runs = dates_df[~dates_df['good_forward']]
        num_runs = end_runs.shape[0]
        for j in range(num_runs):
            if j == 0:
                start_ind = 0
            else:
                start_ind = end_runs.index[j-1] + 1
            end_ind = end_runs.index[j]
            jth_run = dates_df.loc[start_ind:end_ind]
            runs_list.append(sorted(jth_run['date']))
        return runs_list


In [113]:
for system_id in enough_data_parquet_power_list[0:10]:
    print(f'System {system_id}')
    major_dir = '../../../../data_ds_project/testing_yearly_parquet/'
    output = good_data_identifier(
        major_dir,
        system_id,
        systems_cleaned,
        return_type='date'
    )
    if output is None:
        print('None')
    elif len(output) == 1:
        print('first reader group')
        for good_run in good_runs(output[0]):
            print(f'From {good_run[0]} to {good_run[-1]}, a total of {len(good_run)} days.')
        print('')
    elif len(output) == 2:
        print('first reader group')
        for good_run in good_runs(output[0]):
            print(f'From {good_run[0]} to {good_run[-1]}, a total of {len(good_run)} days.')
        print('second reader group')
        for good_run in good_runs(output[1]):
            print(f'From {good_run[0]} to {good_run[-1]}, a total of {len(good_run)} days.')
        print('')
    else:
        raise ValueError('Bad output of good_data_identifier!')


System 4
first reader group
From 2007-08-27 to 2007-11-20, a total of 86 days.
From 2007-11-22 to 2007-12-04, a total of 13 days.
From 2007-12-20 to 2008-03-01, a total of 73 days.
From 2008-03-03 to 2008-04-02, a total of 31 days.
From 2008-04-07 to 2008-08-21, a total of 137 days.
From 2008-08-28 to 2008-11-29, a total of 94 days.
From 2008-12-01 to 2008-12-03, a total of 3 days.
From 2008-12-05 to 2009-01-25, a total of 52 days.
From 2009-01-27 to 2009-04-03, a total of 67 days.
From 2009-04-05 to 2009-04-16, a total of 12 days.
From 2009-04-19 to 2009-05-29, a total of 41 days.
From 2010-01-23 to 2010-01-31, a total of 9 days.
From 2010-02-25 to 2010-03-26, a total of 30 days.
From 2010-03-30 to 2010-04-01, a total of 3 days.
From 2010-04-06 to 2010-04-09, a total of 4 days.
From 2010-04-12 to 2010-04-30, a total of 19 days.
From 2010-05-04 to 2010-05-10, a total of 7 days.
From 2010-05-13 to 2010-06-24, a total of 43 days.
From 2010-06-29 to 2010-07-09, a total of 11 days.
From 20

In [112]:
for system_id in (50,):
    print(f'System {system_id}')
    major_dir = '../../../../data_ds_project/testing_yearly_parquet/'
    output = good_data_identifier(
        major_dir,
        system_id,
        systems_cleaned,
        return_type='date'
    )
    if output is None:
        print('None')
    elif len(output) == 1:
        print(len(output[0]))
    elif len(output) == 2:
        print(len(output[0]), len(output[1]))
    else:
        raise ValueError('Bad output of good_data_identifier!')

System 50
7248


Get single-year variant (meant to deal with memory errors before I did something else.)

In [56]:
def good_data_single_year(
    input_dir: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    year: int,
    return_type: str,
):
    '''Grab good data for all years for a given system.
    
    Parameters:
    -------------
    input_dir: str
        The path to the results of ac_power_parquet_distiller_yearly.py
        Default is data_ds_project/testing_yearly_parquet/, 
        but this may change in the final build.
    system_id: int
        The desired system_id.
    systems_cleaned: pd.DataFrame
        The metadata dataframe.
    year: int
        The desired year
    return_type: str, "date" or "time_power"
        If return_type is "date", return a list of dates for each subsystem.
        If return_type is "time_power, return a pandas DataFrame with time/power columns
        and cleaned-data rows
    '''
    reshaped_data = daily_delta_t_reshaper(
        input_dir=input_dir,
        system_id=system_id
    )
    first_timesteps, first_changes, first_bad_dates = reshaped_data[0]
    if len(reshaped_data) == 2:
        second_timesteps, second_changes, second_bad_dates = reshaped_data[1]
    else:
        second_timesteps = None
        second_changes = None
        second_bad_dates = None
    # prepeare return containers
    if return_type == 'date':
        first_cleaned_dates = []
        second_cleaned_dates = []
    elif return_type == 'time_power':
        first_cleaned_compendium = []
        second_cleaned_compendium = []
    else:
        raise ValueError(
            f'return_type is {return_type},'
            + 'not the valid choices of "date" or "time_power".'
        )
    cleaned_data = year_data_extractor(
        input_dir=input_dir,
        system_id=system_id,
        systems_cleaned=systems_cleaned,
        year=year
    )
    if cleaned_data is None:
        return None  # nothing to do
    elif len(cleaned_data) == 1:
        first_cleaned = cleaned_data[0]
        second_cleaned = None
    elif len(cleaned_data) == 2:
        first_cleaned = cleaned_data[0]
        second_cleaned = cleaned_data[1]
    else:
        raise ValueError('Trouble with just_over_year_data_extractor(*args)')
    if (first_cleaned is not None) and (first_cleaned.shape[0] > 10):
        first_cleaned = reconcile_deltas(
            first_cleaned,
            first_timesteps,
            first_bad_dates,
            True
        )
        if first_cleaned is not None:
            if return_type == 'date':
                first_cleaned_dates += sorted(first_cleaned['date'].unique())
            elif return_type == 'time_power':
                first_cleaned_compendium.append(first_cleaned[['time', 'power']])
    if (second_cleaned is not None) and (second_cleaned.shape[0] > 10):
        second_cleaned = reconcile_deltas(
            second_cleaned,
            second_timesteps,
            second_bad_dates,
            True
        )
        if second_cleaned is not None:
            if return_type == 'date':
                second_cleaned_dates += sorted(second_cleaned['date'].unique())
            elif return_type == 'time_power':
                second_cleaned_compendium.append(second_cleaned[['time', 'power']])
    if return_type == 'date':
        if len(first_cleaned_dates) > 0:
            if len(second_cleaned_dates) > 0:
                return (first_cleaned_dates, second_cleaned_dates)
            else:
                return (first_cleaned_dates,)
        elif len(second_cleaned_dates) > 0:  # System 1200, latter years
            return (None, second_cleaned_dates)
        else:
            return None
    elif return_type == 'time_power':
        if len(first_cleaned_compendium) > 0:
            first_cleaned_total = pd.concat(first_cleaned_compendium)
        else:
            # can't raise an error, maybe a bad year
            first_cleaned_total = None
        if len(second_cleaned_compendium) > 0:
            second_cleaned_total = pd.concat(second_cleaned_compendium)
        else:
            second_cleaned_total = None
        if (second_cleaned_total is not None):
            return (first_cleaned_total, second_cleaned_total)
        elif (first_cleaned_total is not None):
            return (first_cleaned_total,)
        else:
            return None
    else:
        return None


Test

In [54]:
for system_id in enough_data_parquet_power_list:
    print(f'System {system_id}')
    num_years = 2023 - 1994 + 1
    # grab number of columns from a dummy year
    input_dir = '../../../../data_ds_project/testing_yearly_parquet/'
    direct_dir = Path(f'../../../../data_ds_project/testing_yearly_parquet/{system_id}/')
    empty_pq = pq.ParquetDataset(
        direct_dir,
        filters = [('year', '==', 1990)]
    )
    empty_df = empty_pq.read().to_pandas()
    num_cols = len(empty_df.columns) - 1  # remove 'year' column
    days_per_year = pd.DataFrame(
        data = np.full(shape=(num_years, num_cols), fill_value=np.nan),
        index = range(1994, 2024),
        columns = ['first_reading', 'second_reading'][0:num_cols]
    )
    for year in range(1994, 2024):
        year_data = good_data_single_year(
            input_dir,
            system_id,
            systems_cleaned,
            year,
            'date'
        )
        if year_data is None:
            continue
        elif len(year_data) == 1:
            days_per_year.at[year, 'first_reading'] = len(year_data[0])
        elif len(year_data) == 2:
            days_per_year.at[year, 'first_reading'] = len(year_data[0])
            days_per_year.at[year, 'second_reading'] = len(year_data[1])
        else:
            raise ValueError(f'For system {system_id}, year {year},\n'
                             + 'the output list from good_data_single_year(*args)'
                             + ' has too long a length.')
    # drop empty rows
    days_per_year = days_per_year.dropna(axis=0, how='all')
    print(days_per_year)
    print('')


System 4


KeyboardInterrupt: 

In [60]:
for system_id in (1200, 4901):
    print(f'System {system_id}')
    num_years = 2023 - 1994 + 1
    # grab number of columns from a dummy year
    input_dir = '../../../../data_ds_project/testing_yearly_parquet/'
    direct_dir = Path(f'../../../../data_ds_project/testing_yearly_parquet/{system_id}/')
    empty_pq = pq.ParquetDataset(
        direct_dir,
        filters = [('year', '==', 1990)]
    )
    empty_df = empty_pq.read().to_pandas()
    num_cols = len(empty_df.columns) - 2  # remove 'time', 'year' columns
    days_per_year = pd.DataFrame(
        data = np.full(shape=(num_years, num_cols), fill_value=np.nan),
        index = range(1994, 2024),
        columns = ['first_reading', 'second_reading'][0:num_cols],
    )
    for year in range(1994, 2024):
        year_data = good_data_single_year(
            input_dir,
            system_id,
            systems_cleaned,
            year,
            'date'
        )
        if year_data is None:
            continue
        elif len(year_data) == 1:
            days_per_year.at[year, 'first_reading'] = len(year_data[0])
        elif len(year_data) == 2:
            if year_data[0] is not None:
                days_per_year.at[year, 'first_reading'] = len(year_data[0])
            if year_data[1] is not None:
                days_per_year.at[year, 'second_reading'] = len(year_data[1])
        else:
            raise ValueError(f'For system {system_id}, year {year},\n'
                             + 'the output list from good_data_single_year(*args)'
                             + ' has too long a length.')
    # drop empty rows
    days_per_year = days_per_year.dropna(axis=0, how='all')
    print(days_per_year)
    print('')

System 1200
      first_reading  second_reading
2010           89.0             NaN
2011          352.0             NaN
2012          234.0             NaN
2013          228.0            32.0
2014            NaN           211.0
2015            NaN           361.0
2016            NaN           361.0
2017            NaN           365.0
2018            NaN           149.0
2019            NaN           358.0
2020            NaN           207.0

System 4901
      first_reading  second_reading
2014          151.0           151.0
2015          357.0           357.0
2016          364.0           364.0
2017          360.0           361.0
2018           71.0            72.0

